# Lesson 3 · Predicting apartment rent

**Question:** How can we use apartment sizes to predict rents we haven't observed?

This is an **optional exercise**. No programming experience is needed. Run the code, change the marked settings, and see what happens.

**How to run:** Select a cell and press **Shift + Enter**. Start at the top. If you change a setting, rerun its cell and any chart below it.

**Data note:** These are *illustrative* apartment observations generated for the exercise—not actual Munich rental records or current rents. They use the same kind of size-versus-rent relationship as the lesson slides.

In [ ]:
%pip install -q numpy pandas matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(12)
area = np.linspace(35, 125, 70)
rent = 50 + 10.5 * area + rng.normal(0, 145, len(area))
apartments = pd.DataFrame({"area_m2": area, "rent_eur": rent})

# Fit the line automatically. We will not calculate the coefficients ourselves.
slope, intercept = np.polyfit(apartments["area_m2"], apartments["rent_eur"], 1)

print(f"Model: predicted monthly rent = €{intercept:.0f} + €{slope:.1f} × area (m²)")
print(f"Observed sizes: {area.min():.0f}–{area.max():.0f} m²")
display(apartments.head())

## Experiment 1 — Make a prediction

Choose an apartment size below. The **slope** tells us how much the prediction changes per additional square meter. The **intercept** is the line's predicted rent at 0 m²—outside our observations, so don't interpret it as a realistic apartment.

In [ ]:
# ===== CHANGE ME =====
APARTMENT_SIZE = 80    # Try 50, 100, or 120

predicted = intercept + slope * APARTMENT_SIZE
print(f"Prediction for {APARTMENT_SIZE} m²: €{predicted:.0f} per month")
print(f"For each additional m², the model predicts €{slope:.1f} more.")

plt.figure(figsize=(8, 4.5))
plt.scatter(area, rent, alpha=0.65, color="#2c8882", label="Observed apartments")
xs = np.linspace(30, 135, 200)
plt.plot(xs, intercept + slope * xs, color="#18313d", linewidth=2, label="Fitted line")
plt.scatter([APARTMENT_SIZE], [predicted], s=115, color="#d27b36", zorder=5, label="Your prediction")
plt.xlabel("Floor area (m²)")
plt.ylabel("Monthly rent (€)")
plt.title("A line gives a prediction for any apartment size")
plt.legend()
plt.tight_layout()
plt.show()

**Think:** Does this predict the exact rent of the next 80 m² apartment? Why might the actual rent differ?

## Experiment 2 — What happens when one observation is unusual?

One apartment has an unusually high rent. We will keep all the other apartments the same, then fit the line again.

A fitted line balances errors across the observations. With **least squares**, one very large error counts especially heavily.

In [ ]:
# ===== CHANGE ME =====
EXTRA_RENT = 2000   # Try 0, 1000, or 4000

changed = apartments.copy()
changed.loc[len(changed) - 3, "rent_eur"] += EXTRA_RENT
new_slope, new_intercept = np.polyfit(changed["area_m2"], changed["rent_eur"], 1)

xs = np.linspace(30, 135, 200)
plt.figure(figsize=(8, 4.5))
plt.scatter(changed["area_m2"], changed["rent_eur"], alpha=0.6, color="#2c8882")
plt.scatter([changed.iloc[-3]["area_m2"]], [changed.iloc[-3]["rent_eur"]],
            color="#d27b36", s=110, zorder=5, label="Changed observation")
plt.plot(xs, intercept + slope * xs, "--", color="#18313d", linewidth=2, label="Original fit")
plt.plot(xs, new_intercept + new_slope * xs, color="#a14f52", linewidth=2, label="Fit after change")
plt.xlabel("Floor area (m²)")
plt.ylabel("Monthly rent (€)")
plt.title("One unusual observation can pull the line")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Original slope: €{slope:.1f}/m²  |  New slope: €{new_slope:.1f}/m²")

**Think:** Is an unusually expensive apartment necessarily a mistake? Should it always be removed?

## Experiment 3 — Interpolation versus extrapolation

The model learned from apartments between **35 and 125 m²**. Try a size inside this range, then one far outside it.

The equation will calculate a number either way. But is that number equally trustworthy?

In [ ]:
# ===== CHANGE ME =====
SIZE_TO_TEST = 250     # Try 80 (interpolation), then 250 (extrapolation)

estimate = intercept + slope * SIZE_TO_TEST
kind = "INTERPOLATION" if area.min() <= SIZE_TO_TEST <= area.max() else "EXTRAPOLATION"
print(f"{SIZE_TO_TEST} m² → €{estimate:.0f} per month ({kind})")

xs = np.linspace(0, max(270, SIZE_TO_TEST + 20), 300)
plt.figure(figsize=(8.5, 4.5))
plt.axvspan(area.min(), area.max(), alpha=0.15, color="#2c8882", label="Observed size range")
plt.scatter(area, rent, alpha=0.6, color="#2c8882", label="Observed apartments")
plt.plot(xs, intercept + slope * xs, color="#18313d", linewidth=2, label="Extended fitted line")
plt.scatter([SIZE_TO_TEST], [estimate], s=120, color="#d27b36", zorder=5, label="Your prediction")
plt.xlabel("Floor area (m²)")
plt.ylabel("Monthly rent (€)")
plt.title("The line continues, but the evidence does not")
plt.legend()
plt.tight_layout()
plt.show()

**Think:** What would have to be true for the model's prediction at 250 m² to be reliable?

### Three ideas to remember
- The slope and intercept describe a prediction rule—not a guarantee.
- Least squares reduces overall squared prediction errors, making very large errors influential.
- Interpolating within observed conditions usually requires fewer assumptions than extrapolating far beyond them.

**Optional extra:** Make `SIZE_TO_TEST` equal to 10. The equation still works. Does that mean the prediction makes sense?